In [7]:
# pip install
#!pip install pandas numpy argon2-cffi


%pip install pandas numpy argon2-cffi

# Used Argon2-cffi to prevent build issues



Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# authentication method

# Import Required Packages
import json
import time
import os
import secrets
import hmac
import hashlib
from argon2 import PasswordHasher
from argon2.exceptions import VerifyMismatchError

# argon2 hasher
ph = PasswordHasher()
#constants
DB_FILE = 'users_database.json'
MAX_ATTEMPTS = 5
LOCKOUT_DURATION = 900
OTP_EXPIRY = 60


#loading user database
def load_user_data():
    if not os.path.exists(DB_FILE):
        print("No user database found.")
        return {} #empty dictionary
    with open(DB_FILE, 'r') as file:
        return json.load(file)


# take user database and dump in JSON file
def save_user_data(data):
    with open(DB_FILE, 'w') as file:
        json.dump(data, file, indent=4)


# save user data in database
def register_user(username, password):
    users = load_user_data()

    #check if user is already registered
    if username in users:
        print("Error: User already exists.")
        return False

    #check if password meets requirements
    if len(password) < 16 or password.lower() == username.lower():
        print("Error: password must be 16+ chars and not your username.")
        return False

    #hash the password
    pw_hash = ph.hash(password)

    #Data chunk for new user to be saved to db
    users[username] = {
        "password": pw_hash,
        "failed_attempts": 0,
        "lockout_until": 0,
        "totp_secret": secrets.token_hex(16),
        "current_otp": None,
        "otp_expiry": 0
    }

    save_user_data(users)
    print(f"User {username} registered successfully.")
    return True


#Verifies user login info and handles lockout scenario
def login_user(username, password):
    users = load_user_data()

    #check if user is in the db
    if username not in users:
        print("User not found.")
        return False

    #find user entry in db and get timestamp
    user = users[username]
    now = time.time()

    #check lockout time (if there is one or if expired)
    if now < user['lockout_until']:
        remaining = int((user['lockout_until'] - now) // 60)
        print(f"Account locked. Try again in {remaining} minutes.")
        return False

    # argon2 constant time comparison
    try:
        ph.verify(user['password'], password)

        # Reset attempts on success
        user['failed_attempts'] = 0
        save_user_data(users)
        return True
    #if failed attempt increment failure and set lockout
    except VerifyMismatchError:
        user['failed_attempts'] += 1
        print(f"Invalid password. Attempt {user['failed_attempts']}/{MAX_ATTEMPTS}")

        if user['failed_attempts'] >= MAX_ATTEMPTS: #five lockout attempts
            user['lockout_until'] = now + LOCKOUT_DURATION #15 minute lockout time
            print("Account locked for 15 minutes.")

        #save to file
        save_user_data(users)
        return False
    #memory cleanup of plaintext pw (just in case)
    finally:
        del password


#generate an OTP for login authentication
def generate_totp(username):
    #grab user information from db
    users = load_user_data()
    user = users[username]

    #create 6 digit otp using HMAC
    msg = str(int(time.time())).encode()
    key = user['totp_secret'].encode()
    h = hmac.new(key, msg, hashlib.sha256).hexdigest()
    otp = str(int(h, 16))[-6:]

    #store in user's entry in db
    user['current_otp'] = otp
    user['otp_expiry'] = time.time() + OTP_EXPIRY
    save_user_data(users)

    #simulate auth app in terminal
    print(f"\n[AUTHENTICATOR APP] Your 2FA Code: {otp} (Expires in 60s)")
    return otp


#verify the user's otp
def verify_totp(username, input_otp):
    #grab user information from db
    users = load_user_data()
    user = users[username]

    #check db entry for otp and expiry time
    if user['current_otp'] is None or time.time() > user['otp_expiry']:
        print("OTP has expired or does not exist.")
        return False

    #constant-time comparison for freshness
    if hmac.compare_digest(user['current_otp'], input_otp):
        #erase/override current otp so it cannot be used again
        user['current_otp'] = None
        save_user_data(users)
        return True
    else:
        print("Invalid OTP.")
        return False

In [9]:
# main

def print_menu():
    """Display the main menu options."""
    print("\n" + "="*50)
    print("        AUTHENTICATION SYSTEM")
    print("="*50)
    print("1. Register a new user")
    print("2. Log in")
    print("3. View User Data")
    print("4. Exit")
    print("="*50)


def main():
    # Main menu handler
    while True:
        print_menu()
        choice = input("Enter your choice (1-4): ").strip()

        # Option 1: Register
        if choice == "1":
            print("\n--- REGISTER ---")
            try:
                username = input("Enter username: ").strip()
                if not username:
                    print("Error: Username cannot be empty.")
                    continue

                password = input("Enter password: ").strip()
                if not password:
                    print("Error: Password cannot be empty.")
                    continue

                register_user(username, password)

            # Standard error handling
            except KeyboardInterrupt:
                print("\nRegistration cancelled.")
            except Exception as e:
                print(f"An error occurred during registration: {e}")

        # Option 2: Login
        elif choice == "2":
            print("\n--- LOGIN ---")
            try:
                # Input validation
                username = input("Enter username: ").strip()
                if not username:
                    print("Error: Username cannot be empty.")
                    continue

                password = input("Enter password: ").strip()
                if not password:
                    print("Error: Password cannot be empty.")
                    continue

                # Verify if creds are okay before considering OTP
                if login_user(username, password):
                    print("Password verified. Generating OTP...")
                    otp = generate_totp(username)

                    # Now ask for OTP
                    input_otp = input("Enter the OTP from your authenticator app: ").strip()
                    if verify_totp(username, input_otp):
                        print("OTP validated. Welcome to the system.")
                    else:
                        print("OTP validation failed. Access denied.")
                else:
                    print("Login failed. Please try again.")

            # Standard error handling
            except KeyboardInterrupt:
                print("\nLogin cancelled.")
            except Exception as e:
                print(f"An error occurred during login: {e}")



        # Option 3: View User Data
        elif choice == "3":
            print("\n--- VIEW USER DATA ---")
            try:
                json_is_not_a_database = load_user_data()
                if not json_is_not_a_database:
                    print("No user data found.")
                    continue
                print("User data found.")
                for username, data in json_is_not_a_database.items():
                    print(f"Username: {username}")
                    print(f"  Failed Attempts: {data['failed_attempts']}")
                    print(f"  Lockout Until: {time.ctime(data['lockout_until'])}")
                    print(f"  TOTP Secret: {data['totp_secret']}")
                    print(f"  Current OTP: {data['current_otp']}")
                    print(f"  OTP Expiry: {time.ctime(data['otp_expiry'])}")
                    # Alternatively
                    #print(json_is_not_a_database)
            except KeyboardInterrupt:
                print("\nViewing user data cancelled.")
            except Exception as e:
                print(f"An error occurred while viewing user data: {e}")


        # Option 4: Exit
        elif choice == "4":
            print("Exiting the system. Goodbye!")
            break
        else:
            print("Invalid choice. Please enter a number between 1 and 4.")


if __name__ == "__main__":
    main()


        AUTHENTICATION SYSTEM
1. Register a new user
2. Log in
3. View User Data
4. Exit

--- VIEW USER DATA ---
User data found.
Username: vsl
  Failed Attempts: 0
  Lockout Until: Wed Dec 31 19:00:00 1969
  TOTP Secret: 76519d67282a3697cd649bc39c397237
  Current OTP: 614830
  OTP Expiry: Wed Apr  1 21:57:18 2026
{'vsl': {'password': '$argon2id$v=19$m=65536,t=3,p=4$Y/Rbe855+5iC43+zTGYfLw$4NVDCuHNQYBL+1SXewth+xYvhuPLe1n9S12NtOqg830', 'failed_attempts': 0, 'lockout_until': 0, 'totp_secret': '76519d67282a3697cd649bc39c397237', 'current_otp': '614830', 'otp_expiry': 1775095038.6227677}}

        AUTHENTICATION SYSTEM
1. Register a new user
2. Log in
3. View User Data
4. Exit


KeyboardInterrupt: Interrupted by user